In [1]:
from utils import build_electrode, build_nanoribbon
from utils.plots import mark_electrode, plot_with_center
import numpy as np
width = 7 # number of atoms in y for a armchair configuration
length = 2 # how many 'armchairs' is tiled in x
center_size = 2 # how many 'electrodes' to use for center region
electrode = build_electrode(width, length)
ribbon = build_nanoribbon(electrode, center_size)

def shift_rotation_center(coords, rotation_center):
    atol = 0.99
    bond_length = 1.42  # approximate C-C bond length in angstroms
    inner_radii = bond_length * np.cos(np.pi / 6)  # distance from center to side of hexagon
    dist = np.linalg.norm(coords - rotation_center, axis=1)
    if np.any(dist <= bond_length*atol): # distance from center to corner of hexagon = sidelength
        print("Warning rotation ")
        print(np.argwhere(dist <= bond_length))
        rotation_center += np.array([0, inner_radii, 0])

    return rotation_center

def _old_build_device_routine(ribbon, electrode, repeat=3):
    from utils.structure import guess_hexagon_center, find_electrode_indices, find_nearest_atoms
    from utils.structure import _mark_electrode, remove_overlaps, _reset_atoms
    coords = ribbon.xyz
    geom_center = ribbon.center()
    center_atoms = find_nearest_atoms(ribbon.xyz, geom_center, neighbours=6)
    
    origin_of_rotation = coords[center_atoms].mean(axis=0)
    origin_of_rotation = shift_rotation_center(coords[center_atoms], origin_of_rotation)
    
    print("Rotation center:", origin_of_rotation)
    print("Geomtric center:", geom_center)
    nanoribbon = ribbon.copy()
    _mark_electrode(nanoribbon, electrode) # mark electrodes with special atoms 
    device = nanoribbon.copy()
    rotation_axis = [0, 0, 1]  # z-axis
    for i in range(1, repeat):
        a = 60
        angle = a*(-1) if i%2 == 0 else a
        rotated_nanoribbon = nanoribbon.rotate(angle=angle, v=rotation_axis, origin=origin_of_rotation)
        device += rotated_nanoribbon
    device = remove_overlaps(device)
    left_idx, right_idx = find_electrode_indices(device, repeat) # find indices where atoms are marked as electrodes
    _reset_atoms(device) # reset all special electrode atoms to normal carbon atoms
    device.set_nsc((1,1,1))
    return device, (left_idx, right_idx)

device, lr_idx = _old_build_device_routine(ribbon, electrode, 3)
a_style = []
for arm in range(3):
    l_style = {"atoms": lr_idx[0][arm], "color": "blue", "size": arm*0.3 + 0.5}
    r_style = {"atoms": lr_idx[1][arm], "color": "red", "size": arm*0.3 + 0.5}
    a_style.append(l_style)
    a_style.append(r_style)
device.plot(axes="xy", atoms_style=a_style)

Warning rotation 
[[0]
 [1]]
Rotation center: [16.33        6.41902429  1.5       ]
Geomtric center: [16.33        5.18926822  1.5       ]
       Initial number of atoms: 336
 Atoms after removing overlaps: 262
